# A contaminant plume on a freshwater island

A landfill leaks into the middle of an island. The island is surrounded by the
sea, recharge falls on it, and groundwater flows outward from the water-table
mound in the center to the shore. The solute leaving the landfill has to travel
with that flow, and the question a hydrologist is asked is where it goes, how
fast, and how much of it arrives.

## What this notebook covers

Build a two-dimensional flow and transport model of that island on an
unstructured grid, and follow the plume for a century.

By the end of this notebook you will be able to:

- build a Voronoi grid over an irregular boundary with Triangle,
- run a transport model from saved flow output through the Flow Model Interface
  (**FMI**), rather than solving flow and transport together,
- place a constant-concentration source with `flopy.utils.GridIntersect`, and
- read a plume from log-scaled concentration contours and step through it in
  time.

The model is in meters and days, as MODFLOW 6 has no units of its own.

Import the packages this notebook uses. `%matplotlib inline` comes first: without
it only the first figure drawn inside a `flopy.plot.styles` context appears.

In [ ]:
%matplotlib inline

import pathlib as pl

import flopy
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from flopy.utils.postprocessing import get_specific_discharge
from flopy.utils.triangle import Triangle
from flopy.utils.voronoi import VoronoiGrid
from mf6_notebook_helpers import find_mf6_libraries, time_slider_view
from shapely.geometry import Point, Polygon

_, mf6_exe = find_mf6_libraries()

## Mesh the island

The island is a 17-sided polygon about 9.3 km across and 6.7 km from north to
south. An unstructured grid follows that shape without the stair-stepped edge a
rectangular grid would give it.

Build the mesh in two steps. Triangle fills the polygon with triangles no larger
than `maximum_area` and with no angle smaller than 30 degrees, and
`VoronoiGrid()` turns that triangulation into the Voronoi polygons MODFLOW 6
uses as cells in a discretization-by-vertices (**DISV**) grid. Cells on the
boundary are found from the triangulation's edges, because those are where the
sea is.

In [ ]:
model_ws = pl.Path("models/gwt-plume")
name = "island"

domain = Polygon(
    [
        [1831.38, 6335.54],
        [4337.73, 6851.13],
        [6428.74, 6707.91],
        [8662.98, 6493.08],
        [9350.43, 5891.56],
        [9235.86, 4717.15],
        [8963.74, 3685.97],
        [8691.62, 2783.68],
        [8047.13, 2038.94],
        [7416.96, 578.09],
        [6414.42, 105.46],
        [5354.59, 205.72],
        [4624.17, 363.26],
        [3363.83, 563.77],
        [1330.11, 1809.78],
        [399.18, 2998.51],
        [914.77, 5132.49],
    ]
)

triangle_ws = model_ws / "_triangle"
triangle_ws.mkdir(parents=True, exist_ok=True)
tri = Triangle(maximum_area=100.0**2, angle=30, model_ws=triangle_ws)
tri.add_polygon(domain)
tri.build(verbose=False)
voronoi_obj = VoronoiGrid(tri)
grid_properties = voronoi_obj.get_disv_gridprops()

# every vertex on a triangulation edge is on the shore, and those cells get the
# sea. Asking for the cells along one edge is what fills tri.edgedict.
tri.get_edge_cells(0)
on_shore = np.zeros(tri.nvert, dtype=int)
for iv1, iv2 in tri.edgedict:
    on_shore[iv1] = 1
    on_shore[iv2] = 1

print(f"triangles:    {tri.ncpl:,}")
print(f"voronoi cells: {grid_properties['ncpl']:,}")
print(f"shore cells:   {on_shore.sum():,}")

## Run the steady flow model

One layer, from 50 m at the top to -100 m at the bottom, with the water table
free to move because `icelltype=1` makes the layer convertible. Recharge of
0.001 m/d falls on every cell, and a constant head (**CHD**) of 1.0 m around the
shore holds the island at sea level. The balance of the two sets how high the
water table mounds in the middle.

Ask the Node Property Flow (**NPF**) package to save specific discharge, which is
what the flow arrows in the figure are drawn from, and save head and budget for
the transport model to read later.

In [ ]:
flow_ws = model_ws / "flow"
sim = flopy.mf6.MFSimulation(sim_name=name, sim_ws=flow_ws, exe_name=str(mf6_exe))
flopy.mf6.ModflowTdis(sim)
flopy.mf6.ModflowIms(sim, inner_maximum=100)
gwf = flopy.mf6.ModflowGwf(sim, modelname=name, save_flows=True)
flopy.mf6.ModflowGwfdisv(gwf, nlay=1, **grid_properties, top=50.0, botm=[-100.0])
flopy.mf6.ModflowGwfic(gwf, strt=50.0)
flopy.mf6.ModflowGwfnpf(
    gwf, icelltype=1, save_specific_discharge=True, save_saturation=True
)
flopy.mf6.ModflowGwfchd(
    gwf, stress_period_data=[[(0, j), 1.0] for j in np.where(on_shore == 1)[0]]
)
flopy.mf6.ModflowGwfrcha(gwf, recharge=0.001)
flopy.mf6.ModflowGwfoc(
    gwf,
    budget_filerecord=f"{name}.bud",
    head_filerecord=f"{name}.hds",
    saverecord=[("HEAD", "ALL"), ("BUDGET", "ALL")],
)
sim.write_simulation(silent=True)
success, buff = sim.run_simulation(silent=True)
if not success:
    raise RuntimeError("\n".join(buff[-20:]))

head = gwf.output.head().get_data()
spdis = gwf.output.budget().get_data(text="DATA-SPDIS")[0]
qx, qy, _ = get_specific_discharge(spdis, gwf)
print(f"water table: {head.min():.2f} to {head.max():.2f} m")

Map the flow. Panel A is the island in plan view, with the shore cells, the water
table contoured, and the flow direction at every cell; panel B is a section along
y = 3800 m, through the middle of the island.

In [ ]:
section_y = 3800.0

with flopy.plot.styles.USGSMap():
    fig, axd = plt.subplot_mosaic(
        [["A"], ["B"]], figsize=(7.0, 8.0), layout="constrained"
    )

    ax = axd["A"]
    pmv = flopy.plot.PlotMapView(gwf, ax=ax)
    pmv.plot_grid(linewidth=0.3, color="0.85")
    pmv.plot_bc(ftype="CHD", color="tab:cyan")
    pmv.plot_vector(qx, qy, normalize=True, color="0.3", width=0.002)
    contours = pmv.contour_array(head, colors="tab:blue", linewidths=0.8)
    ax.clabel(contours, fmt="%.1f", fontsize=7)
    ax.plot(*domain.exterior.xy, color="k", lw=2.0)
    ax.plot([0.0, 9500.0], [section_y, section_y], "k--", lw=1.0)
    ax.set_aspect(1.0)
    ax.set_xlabel("x, in meters")
    ax.set_ylabel("y, in meters")
    ax.set_title("A. Water table and flow direction")

    ax = axd["B"]
    pxs = flopy.plot.PlotCrossSection(
        gwf, ax=ax, line={"line": [(0.0, section_y), (9500.0, section_y)]}
    )
    pxs.plot_array(head, head=head, cmap="Blues")
    pxs.plot_bc(ftype="CHD", color="tab:cyan")
    pxs.plot_grid(color="0.6", linewidth=0.3)
    ax.set_aspect(10.0)
    ax.set_xlabel("x, in meters")
    ax.set_ylabel("elevation, in meters")
    ax.set_title(f"B. Section along y = {section_y:,.0f} m")

**What to look for.** The water table mounds to 30.6 m in the middle of the
island and falls to the 1.0 m the sea holds at the shore, so the arrows point
outward in every direction from the mound and the contours close on themselves.
There is no single downstream direction here, which is what makes the plume's
path worth simulating: where the solute goes depends on which part of the mound
it starts from. The section in panel B is the same mound seen edge on, 30 m of
relief over an aquifer 150 m thick.

## Add the contaminant source

Transport runs as its own simulation, reading the flow it needs from the head and
budget files the flow model just wrote. That is what the Flow Model Interface
(**FMI**) package does, and it is the cheaper way to work when the flow field
does not depend on the solute: the flow model runs once, and any number of
transport runs can follow it.

The source is a constant concentration (**CNC**) of 100 mg/L in one cell, which
represents something that keeps leaking at the same strength however much water
carries it away. `GridIntersect` finds which Voronoi cell contains the point,
since on an unstructured grid there is no row and column to count.

Transport also needs the properties advection and dispersion depend on: a
porosity of 0.25, upstream advection, and dispersivities of 1.0 m along the flow
direction and 0.1 m across it.

In [ ]:
transport_ws = model_ws / "transport"
source_point = Point(6000.0, 4000.0)

sim_gwt = flopy.mf6.MFSimulation(
    sim_name=name, sim_ws=transport_ws, exe_name=str(mf6_exe)
)
flopy.mf6.ModflowTdis(sim_gwt, nper=1, perioddata=[(36500.0, 100, 1.0)])
flopy.mf6.ModflowIms(sim_gwt, linear_acceleration="BICGSTAB", inner_maximum=100)
gwt = flopy.mf6.ModflowGwt(sim_gwt, modelname=name, save_flows=True)
flopy.mf6.ModflowGwtdisv(gwt, nlay=1, **grid_properties, top=50.0, botm=[-100.0])
flopy.mf6.ModflowGwtic(gwt, strt=0.0)
flopy.mf6.ModflowGwtmst(gwt, porosity=0.25)
flopy.mf6.ModflowGwtadv(gwt, scheme="upstream")
flopy.mf6.ModflowGwtdsp(gwt, alh=1.0, ath1=0.1)
flopy.mf6.ModflowGwtssm(gwt, sources=[()])
flopy.mf6.ModflowGwtfmi(
    gwt,
    packagedata=[
        ("GWFHEAD", f"../flow/{name}.hds"),
        ("GWFBUDGET", f"../flow/{name}.bud"),
    ],
)

source_cell = flopy.utils.GridIntersect(gwt.modelgrid).intersect(
    source_point, geo_dataframe=False
)["cellids"][0]
flopy.mf6.ModflowGwtcnc(
    gwt, stress_period_data=[((0, source_cell), 100.0)], pname="cnc-1"
)
flopy.mf6.ModflowGwtoc(
    gwt,
    budget_filerecord=f"{name}.cbc",
    concentration_filerecord=f"{name}.ucn",
    budgetcsv_filerecord=f"{name}-budget.csv",
    saverecord=[("CONCENTRATION", "ALL"), ("BUDGET", "ALL")],
)
sim_gwt.write_simulation(silent=True)
success, buff = sim_gwt.run_simulation(silent=True)
if not success:
    raise RuntimeError("\n".join(buff[-20:]))

conc_obj = gwt.output.concentration()
conc = conc_obj.get_alldata()
times = np.array(conc_obj.times)
print(f"source cell:   {source_cell}")
print(f"output times:  {len(times)}, ending at {times[-1] / 365.25:.0f} years")

Plot the plume at the end of the century. Concentration spans five orders of
magnitude, so the contours are log-spaced from 0.001 to 100 mg/L; anything below
the lowest contour is left white.

In [ ]:
levels = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
contour_style = {
    "cmap": plt.colormaps["viridis"].with_extremes(under="white", over="white"),
    "extend": "both",
    "levels": levels,
    "filled": True,
    "norm": "log",
}

with flopy.plot.styles.USGSMap():
    fig, ax = plt.subplots(figsize=(7.0, 5.0), layout="constrained")
    pmv = flopy.plot.PlotMapView(gwt, ax=ax)
    pmv.plot_grid(linewidth=0.2, color="0.9")
    pmv.contour_array(head, colors="0.4", linewidths=0.6)
    plume = pmv.contour_array(conc[-1], **contour_style)
    ax.plot(*domain.exterior.xy, color="k", lw=2.0)
    ax.plot(source_point.x, source_point.y, "r*", ms=12)
    ax.set_aspect(1.0)
    ax.set_xlabel("x, in meters")
    ax.set_ylabel("y, in meters")
    ax.set_title("Plume after 100 years")
    fig.colorbar(
        plume, ax=ax, shrink=0.6, label="concentration, in milligrams per liter"
    )

**What to look for.** The plume leaves the source (red star) and travels east,
toward the nearest shore, because that is the way the water table falls there. A
century on it is a broad lobe some 2.5 km across, reaching the 0.001 mg/L contour
almost to the eastern shoreline while the 10 mg/L contour has moved only a few
hundred meters. The spreading is mostly the flow field's doing rather than
dispersion: the arrows in the previous figure fan outward from the mound, so
neighboring streamlines carry the solute apart as it goes. The gray lines are
water-table contours, and the plume crosses them at right angles, which is what
solute following the flow has to do.

## Follow the plume through time

The transport model saved a concentration array at every one of the 100 time
steps. Step through them with the slider; each frame is the plume at one year.

In [ ]:
def plume_frame(index):
    """Draw the plume at one output time."""
    with flopy.plot.styles.USGSMap():
        fig, ax = plt.subplots(figsize=(7.0, 5.0), layout="constrained")
        pmv = flopy.plot.PlotMapView(gwt, ax=ax)
        pmv.plot_grid(linewidth=0.2, color="0.9")
        pmv.contour_array(head, colors="0.4", linewidths=0.6)
        pmv.contour_array(conc[index], **contour_style)
        ax.plot(*domain.exterior.xy, color="k", lw=2.0)
        ax.plot(source_point.x, source_point.y, "r*", ms=12)
        ax.set_aspect(1.0)
        ax.set_xlabel("x, in meters")
        ax.set_ylabel("y, in meters")
        ax.set_title(f"Year {times[index] / 365.25:.0f}")
    return fig


time_slider_view(plume_frame, len(times), description="year")

## Two questions the model can answer

A plume map is a picture; the questions asked of a transport model are usually
numbers. Take two.

**When does a point downgradient first see the solute?** Read the concentration
history in the cell at (7000, 4500), about 1.1 km northeast of the source, and
find when it passes 0.1 mg/L.

**How much solute entered the aquifer?** A constant-concentration cell supplies
whatever mass the flow through it carries, so the amount is an output of the
model rather than something specified. It is in the transport budget.

In [ ]:
observation_point = Point(7000.0, 4500.0)
observation_cell = flopy.utils.GridIntersect(gwt.modelgrid).intersect(
    observation_point, geo_dataframe=False
)["cellids"][0]
history = conc[:, 0, 0, observation_cell]

threshold = 0.1
above = np.flatnonzero(history > threshold)
arrival = times[above[0]] / 365.25
print(f"concentration at {observation_point.x:,.0f}, {observation_point.y:,.0f} m")
print(f"  passes {threshold} mg/L after {arrival:.0f} years")
print(f"  ends the run at {history[-1]:.2f} mg/L")

budget = pd.read_csv(transport_ws / f"{name}-budget.csv")
source_in = [c for c in budget.columns if "CNC" in c.upper() and c.endswith("_IN")][0]
mass = np.trapezoid(budget[source_in], budget["time"])
print()
print(f"solute mass from the source over 100 years: {mass / 1.0e6:,.0f} kg")

**What to look for.** The point 1.1 km from the source waits 63 years to pass
0.1 mg/L, and ends the century at 1.14 mg/L, still a hundredth of the source
strength. Arrival is gradual rather than a front sweeping past, and a monitoring
well there would read nothing at all for the first half of the simulation.

The source delivered 271 kg of solute over the century. Nothing in the input says
so: a constant-concentration cell supplies whatever mass the water passing
through it can carry, so doubling the recharge, or moving the source to a faster
part of the flow field, would change that number without changing a single
concentration in the input.

## Recap

- An irregular domain is meshed by triangulating it and building Voronoi cells
  from the triangulation, which gives a **DISV** grid that follows the shoreline.
- Recharge on an island with the sea held at its edge makes a water-table mound,
  so flow leaves in every direction and the plume's path depends on where the
  source sits on that mound.
- A transport model can read its flow from saved head and budget files through
  the **FMI** package, so one flow run can serve many transport runs.
- `GridIntersect` turns a coordinate into a cell, which is how boundaries are
  placed on an unstructured grid.
- Log-spaced contours are the way to look at a plume, because concentration
  spans orders of magnitude between the source and the leading edge.